In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin


# =========================
# 3. FEATURES SELECTION
# =========================

target = '0a. Has committed ESBs?'

categorical_features = [
    '1p. Locale broad type (name)',
    '1q. Census Region'
]

numeric_features = [
    '2a. Total number of buses',
    '4b. Number of students in district',
    '4c. Number of schools in district',
    '4f. Median household income',
    '4g. Percent of population below the poverty level',
    '5f. PM2.5 concentration',
    '5h. Ozone concentration'
]

features = categorical_features + numeric_features

path = "../data/TP1/"

X_train= pd.read_csv(f"{path}/X_train.csv")
X_test = pd.read_csv(f"{path}/X_test.csv")
y_test = pd.read_csv(f"{path}/y_test.csv")
y_train = pd.read_csv(f"{path}/y_train.csv")

class MultiTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}

    def fit(self, X, y):
        X = pd.DataFrame(X)
        for col in X.columns:
            enc = TargetEncoder()
            enc.fit(X[col], y)
            self.encoders[col] = enc
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            X[col] = self.encoders[col].transform(X[col])
        return X


# =========================
# 7. PREPROCESSING
# =========================

preprocess = ColumnTransformer(
    transformers=[
        ('cat', MultiTargetEncoder(), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)


# =========================
# 8. MODEL
# =========================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42
)


# =========================
# 9. PIPELINE (SMOTE INCLUDED)
# =========================

pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', model)
])


# =========================
# 10. TRAINING
# =========================

pipeline.fit(X_train, y_train)


# =========================
# 11. PREDICTION
# =========================

y_pred = pipeline.predict(X_test)


# =========================
# 12. EVALUATION
# =========================

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


CONFUSION MATRIX
[[1477  104]
 [ 136   89]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         0.0       0.92      0.93      0.92      1581
         1.0       0.46      0.40      0.43       225

    accuracy                           0.87      1806
   macro avg       0.69      0.66      0.68      1806
weighted avg       0.86      0.87      0.86      1806

